# Comparing `pgmuvi`, `celerite2`, and `tinygp` on synthetic light curves

This tutorial responds to the referee request for a more quantitative and practical
comparison between GP tools used in astronomy. We use **the same covariance kernel** in
all three tools and compare:

- how much user code is required,
- how long fitting takes, and
- which tools can natively handle multiwavelength (2D) data.


## Mathematical framework

All three tools model data with a Gaussian process (GP):

$$y(\mathbf{x}) \sim \mathcal{GP}(m(\mathbf{x}),\; k(\mathbf{x},\mathbf{x}')),$$

with a constant mean $m$ and a covariance kernel $k$.

We use the **Spectral Mixture Kernel** (SMK; Wilson & Adams 2013) throughout:

$$k_\mathrm{SM}(\tau) = \sum_{q=1}^{Q} w_q\;\exp\!\bigl(-2\pi^2 v_q^2 \tau^2\bigr)\;\cos(2\pi \mu_q \tau),$$

where $\mu_q$ are mixture-mean frequencies, $v_q$ are bandwidths, and $w_q$ are weights.

**`celerite2`** cannot represent the SMK directly, because its semiseparable structure
requires kernels with *rational power spectra* rather than Gaussian spectra.
The nearest available term is the **SHOTerm** (stochastically driven harmonic oscillator):

$$k_\mathrm{SHO}(\tau;\,Q\!\gg\!\tfrac{1}{2}) \approx\sigma^2\exp\!\Bigl(-\tfrac{\omega_0}{2Q}|\tau|\Bigr)\cos(\omega_0 \tau),$$

which has a **Lorentzian** power spectrum (slower spectral decay) instead of the Gaussian
spectrum of the SMK. Both kernels peak at $\omega_0/(2\pi) = \mu_q$, making the SHOTerm
a valid demonstration substitute while clearly being a different model family.


In [ ]:
import textwrap, time
import numpy as np
import matplotlib.pyplot as plt
import torch

from pgmuvi.synthetic import make_simple_sinusoid_1d, make_chromatic_sinusoid_2d

# --- optional celerite2 ---
try:
    import celerite2
    from celerite2 import terms as c2terms
    from scipy.optimize import minimize
    HAVE_CELERITE = True
except Exception:
    HAVE_CELERITE = False

# --- optional tinygp + JAX + optax ---
try:
    import jax, jax.numpy as jnp
    import equinox as eqx
    import optax
    from tinygp import GaussianProcess as TinyGP
    from tinygp.kernels.base import Kernel as TinyKernel
    HAVE_TINYGP = True
except Exception:
    HAVE_TINYGP = False

print(f'celerite2 available : {HAVE_CELERITE}')
print(f'tinygp    available : {HAVE_TINYGP}')

def count_lines(snippet: str) -> int:
    return sum(1 for ln in textwrap.dedent(snippet).splitlines() if ln.strip())


## Synthetic datasets

We use **`pgmuvi.synthetic`** to generate reproducible synthetic light curves.


In [ ]:
# ---------- 1D single-band light curve ----------
TRUE_PERIOD = 40.0   # days
TRUE_FREQ   = 1.0 / TRUE_PERIOD
Q_MIXTURES  = 2      # same number of components for all tools

lc_1d = make_simple_sinusoid_1d(
    n_obs=120, period=TRUE_PERIOD, amplitude=1.0,
    noise_level=0.2, noise_type='gaussian', irregular=True, seed=1,
)

# ---------- 2D multiwavelength light curve ----------
WAVELENGTHS = [0.5, 0.8, 1.2]   # arbitrary wavelength units
lc_2d = make_chromatic_sinusoid_2d(
    n_per_band=[80, 70, 60],
    period=TRUE_PERIOD,
    wavelengths=WAVELENGTHS,
    amplitude_law='linear',
    amplitude=1.0,
    amplitude_slope=0.6,
    noise_level=0.15,
    t_span=180.0,
    irregular=True,
    seed=2,
)

print('1D dataset :', len(lc_1d.xdata), 'points')
print('2D dataset :', len(lc_2d.xdata), 'points,  ndim =', lc_2d.ndim)


## 1D (single-wavelength) comparison

All three tools can fit a single-band light curve with the SMK (or its approximation).
We use $Q = 2$ mixture components, initialised with the true period and its first harmonic,
so the comparison focuses on **user-code complexity and wall-clock time**, not
on the frequency-search problem.


### `pgmuvi` — 1D spectral mixture GP


In [ ]:
timings = {}

# pgmuvi exposes the SMK directly via the '1D' model shortcut.
# num_mixtures=Q_MIXTURES=2; pgmuvi runs MLS initialisation automatically.
t0 = time.perf_counter()
res_pgmuvi_1d = lc_1d.fit(
    model='1D',
    num_mixtures=Q_MIXTURES,
    training_iter=150,
    miniter=50,
    lr=0.05,
)
timings['pgmuvi_1d'] = time.perf_counter() - t0

print('pgmuvi 1D  | loss:', round(float(res_pgmuvi_1d['loss'][-1]), 3),
      ' | time:', round(timings['pgmuvi_1d'], 2), 's')

# Inspect fitted frequencies
mm = lc_1d.model.covar_module.mixture_means.detach().squeeze().numpy()
print('  fitted frequencies:', mm, '  -> periods:', 1.0 / mm)


### `tinygp` — custom 1D spectral mixture kernel

Because `tinygp` does not ship an SMK, we must define it ourselves — a subclass
of `tinygp.kernels.base.Kernel` that implements **the same formula** used by
`gpytorch.kernels.SpectralMixtureKernel`.


In [ ]:
# ---- custom SMK class for tinygp (1D) ----
# Must be defined before any JAX tracing; works as a standard eqx.Module.

class SMK1D(TinyKernel if HAVE_TINYGP else object):
    r'''1-D Spectral Mixture Kernel.

    k(tau) = sum_q  w_q * cos(2*pi*mu_q*tau) * exp(-2*pi^2*v_q^2*tau^2)

    Parameters are stored in log-space for unconstrained optimisation.
    '''
    log_weights : 'jnp.ndarray'  # (Q,)
    log_freqs   : 'jnp.ndarray'  # (Q,)  mu_q  [cycles / unit]
    log_scales  : 'jnp.ndarray'  # (Q,)  v_q   [cycles / unit]

    def evaluate(self, X1, X2):
        w  = jnp.exp(self.log_weights)
        mu = jnp.exp(self.log_freqs)
        v  = jnp.exp(self.log_scales)
        tau = X1 - X2
        return jnp.sum(
            w * jnp.cos(2 * jnp.pi * mu * tau)
              * jnp.exp(-2 * jnp.pi**2 * v**2 * tau**2)
        )


if HAVE_TINYGP:
    x_j  = jnp.asarray(np.asarray(lc_1d.xdata.detach()))
    y_j  = jnp.asarray(np.asarray(lc_1d.ydata.detach()))
    ye_j = jnp.asarray(np.asarray(lc_1d.yerr.detach()))
    mean1d = float(np.mean(np.asarray(y_j)))

    # Same initialisation as pgmuvi: f0 = TRUE_FREQ, f1 = 2*TRUE_FREQ
    kernel_tiny_1d = SMK1D(
        log_weights=jnp.log(jnp.ones(Q_MIXTURES) * 0.5),
        log_freqs  =jnp.log(jnp.array([TRUE_FREQ, 2 * TRUE_FREQ])),
        log_scales =jnp.log(jnp.ones(Q_MIXTURES) * 0.002),
    )

    @jax.jit
    @jax.value_and_grad
    def _neg_ll_tiny1d(params):
        gp = TinyGP(params, x_j, diag=ye_j**2, mean=mean1d)
        return -gp.log_probability(y_j)

    opt   = optax.adam(0.05)
    state = opt.init(eqx.filter(kernel_tiny_1d, eqx.is_inexact_array))

    t0 = time.perf_counter()
    for _ in range(150):
        loss, grads = _neg_ll_tiny1d(kernel_tiny_1d)
        updates, state = opt.update(
            grads, state, eqx.filter(kernel_tiny_1d, eqx.is_inexact_array))
        kernel_tiny_1d = eqx.apply_updates(kernel_tiny_1d, updates)
    timings['tinygp_1d'] = time.perf_counter() - t0

    freqs_fit = np.exp(np.asarray(kernel_tiny_1d.log_freqs))
    print('tinygp 1D  | loss:', round(float(loss), 3),
          ' | time:', round(timings['tinygp_1d'], 2), 's')
    print('  fitted frequencies:', freqs_fit, '  -> periods:', 1.0 / freqs_fit)
else:
    print('tinygp not installed.')


### `celerite2` — SHOTerm approximation of the SMK

celerite2 uses a semiseparable matrix representation that restricts kernels to those
with **rational power spectra** (Lorentzian, not Gaussian shape). The SHOTerm is the
closest available single-frequency term:

$$k_\mathrm{SHO}(\tau) \approx \sigma^2\exp\!\Bigl(-\tfrac{\omega_0}{2Q}|\tau|\Bigr)\cos(\omega_0\tau), \quad Q \gg \tfrac12,$$

with $\omega_0 = 2\pi/\rho$ (period $\rho$) and quality factor $Q = \omega_0\tau/2$.
We use one `SHOTerm` per SMK mixture component.


In [ ]:
# ---- celerite2 1D fit with 2-component SHO kernel ----

if HAVE_CELERITE:
    x_c  = np.asarray(lc_1d.xdata.detach())
    y_c  = np.asarray(lc_1d.ydata.detach())
    ye_c = np.asarray(lc_1d.yerr.detach())
    mean1d_c = float(np.mean(y_c))

    # Initial parameters: match SMK init (rho = period, tau = damping)
    def _build_c2_kernel_1d(p):
        # p = [log_sigma1, log_rho1, log_tau1, log_sigma2, log_rho2, log_tau2]
        return (
            c2terms.SHOTerm(
                sigma=np.exp(p[0]), rho=np.exp(p[1]), tau=np.exp(p[2])) +
            c2terms.SHOTerm(
                sigma=np.exp(p[3]), rho=np.exp(p[4]), tau=np.exp(p[5]))
        )

    def _neg_ll_c2_1d(p):
        try:
            gp_ = celerite2.GaussianProcess(_build_c2_kernel_1d(p), mean=mean1d_c)
            gp_.compute(x_c, yerr=ye_c)
            return -gp_.log_likelihood(y_c)
        except Exception:
            return 1e10

    # Same period initialisation as the other tools
    p0 = [
        np.log(0.7), np.log(TRUE_PERIOD),     np.log(2 * TRUE_PERIOD),
        np.log(0.3), np.log(TRUE_PERIOD / 2), np.log(TRUE_PERIOD),
    ]

    t0  = time.perf_counter()
    res = minimize(_neg_ll_c2_1d, p0, method='L-BFGS-B',
                   options={'maxiter': 150})
    timings['celerite2_1d'] = time.perf_counter() - t0

    rho1 = np.exp(res.x[1])   # fitted period component 1
    rho2 = np.exp(res.x[4])   # fitted period component 2
    print('celerite2 1D | loss:', round(res.fun, 3),
          ' | time:', round(timings['celerite2_1d'], 3), 's')
    print(f'  fitted SHO periods: {rho1:.2f}, {rho2:.2f}  (true: {TRUE_PERIOD})')
else:
    print('celerite2 not installed.')


### Code-line comparison for 1D fitting


In [ ]:
pgmuvi_1d_code = '''
res = lc_1d.fit(model='1D', num_mixtures=2, training_iter=150, lr=0.05)
'''

tinygp_1d_code = '''
# (1) define custom kernel class once (reusable)
class SMK1D(TinyKernel):
    log_weights: jnp.ndarray
    log_freqs:   jnp.ndarray
    log_scales:  jnp.ndarray
    def evaluate(self, X1, X2):
        w, mu, v = map(jnp.exp, (self.log_weights, self.log_freqs, self.log_scales))
        tau = X1 - X2
        return jnp.sum(w * jnp.cos(2*jnp.pi*mu*tau) * jnp.exp(-2*jnp.pi**2*v**2*tau**2))
# (2) instantiate + optimise
kernel = SMK1D(log_weights=..., log_freqs=..., log_scales=...)
@jax.jit
@jax.value_and_grad
def neg_ll(k): return -TinyGP(k, x, diag=yerr**2).log_probability(y)
# ... 3-line optax loop ...
'''

celerite2_1d_code = '''
# SHOTerm is a Lorentzian approximation to the SMK (see text)
kernel = SHOTerm(sigma=s1, rho=rho1, tau=tau1) + SHOTerm(sigma=s2, rho=rho2, tau=tau2)
gp = celerite2.GaussianProcess(kernel, mean=mean_val)
gp.compute(x, yerr=yerr)
result = minimize(neg_ll, p0, method='L-BFGS-B')
'''

print(f"Code lines (1D fitting):")
print(f"  pgmuvi    :  {count_lines(pgmuvi_1d_code):2d}  (including kernel definition)")
print(f"  tinygp    :  {count_lines(tinygp_1d_code):2d}  (kernel class + optimiser loop)")
print(f"  celerite2 :  {count_lines(celerite2_1d_code):2d}  (SHO proxy for SMK + scipy)")
print()
print('Wall-clock times:')
for k,v in timings.items():
    print(f'  {k:<20s}: {v:.3f} s')


## 2D (multiwavelength) comparison

Here the differences are clearest. `pgmuvi` has a native 2D SMK shortcut.
`tinygp` requires a hand-written 2D kernel class.
`celerite2` is restricted to 1D time-series and cannot accept `(time, wavelength)` inputs.


### `pgmuvi` — 2D spectral mixture GP


In [ ]:
t0 = time.perf_counter()
res_pgmuvi_2d = lc_2d.fit(
    model='2D',
    num_mixtures=Q_MIXTURES,
    training_iter=150,
    miniter=50,
    lr=0.05,
)
timings['pgmuvi_2d'] = time.perf_counter() - t0

print('pgmuvi 2D  | loss:', round(float(res_pgmuvi_2d['loss'][-1]), 3),
      ' | time:', round(timings['pgmuvi_2d'], 2), 's')

mm2 = lc_2d.model.covar_module.mixture_means.detach()
time_freqs = mm2[:, 0, 0].numpy()
print('  fitted time frequencies:', time_freqs, '-> periods:', 1.0 / time_freqs)


### `tinygp` — custom 2D spectral mixture kernel

The 2D SMK is a **product over dimensions** of per-dimension spectral sums:

$$k_\mathrm{SM}^{(2D)}(\mathbf{x},\mathbf{x}') =\prod_{d=1}^{2}\sum_{q=1}^{Q} w_q\exp\!\bigl(-2\pi^2 v_q^{(d)2}\tau_d^2\bigr)\cos(2\pi\mu_q^{(d)}\tau_d).$$

This must be defined explicitly in `tinygp` — it is built into `pgmuvi`.


In [ ]:
# ---- custom 2D SMK for tinygp ----

class SMK2D(TinyKernel if HAVE_TINYGP else object):
    r'''2-D Spectral Mixture Kernel (product over dimensions, sum over mixtures).

    k(x1, x2) = prod_{d in {time, wl}} sum_q  w_q
                * cos(2*pi*mu_q^d*(x1_d-x2_d))
                * exp(-2*pi^2*v_q^d^2*(x1_d-x2_d)^2)
    '''
    log_weights : 'jnp.ndarray'   # (Q,)
    log_freqs   : 'jnp.ndarray'   # (Q, 2)
    log_scales  : 'jnp.ndarray'   # (Q, 2)

    def evaluate(self, X1, X2):
        w  = jnp.exp(self.log_weights)
        mu = jnp.exp(self.log_freqs)
        v  = jnp.exp(self.log_scales)
        tau = X1 - X2  # (2,)

        def _dim(d):
            return jnp.sum(
                w * jnp.cos(2 * jnp.pi * mu[:, d] * tau[d])
                  * jnp.exp(-2 * jnp.pi**2 * v[:, d]**2 * tau[d]**2)
            )

        return _dim(0) * _dim(1)


if HAVE_TINYGP:
    x2_j  = jnp.asarray(np.asarray(lc_2d.xdata.detach()))
    y2_j  = jnp.asarray(np.asarray(lc_2d.ydata.detach()))
    ye2_j = jnp.asarray(np.asarray(lc_2d.yerr.detach()))
    mean2d = float(np.mean(np.asarray(y2_j)))

    # Initialise: time freq = TRUE_FREQ, wl freq = 1/wl_range
    wl_range  = max(WAVELENGTHS) - min(WAVELENGTHS)
    wl_freq   = 1.0 / wl_range
    init_freqs_2d = jnp.array(
        [[TRUE_FREQ,     wl_freq],
         [2 * TRUE_FREQ, 2 * wl_freq]]
    )
    kernel_tiny_2d = SMK2D(
        log_weights=jnp.log(jnp.ones(Q_MIXTURES) * 0.5),
        log_freqs  =jnp.log(init_freqs_2d),
        log_scales =jnp.log(jnp.ones((Q_MIXTURES, 2)) * 0.002),
    )

    @jax.jit
    @jax.value_and_grad
    def _neg_ll_tiny2d(params):
        gp = TinyGP(params, x2_j, diag=ye2_j**2, mean=mean2d)
        return -gp.log_probability(y2_j)

    opt2d   = optax.adam(0.05)
    state2d = opt2d.init(eqx.filter(kernel_tiny_2d, eqx.is_inexact_array))

    t0 = time.perf_counter()
    for _ in range(150):
        loss2d, grads2d = _neg_ll_tiny2d(kernel_tiny_2d)
        upd2d, state2d  = opt2d.update(
            grads2d, state2d,
            eqx.filter(kernel_tiny_2d, eqx.is_inexact_array))
        kernel_tiny_2d = eqx.apply_updates(kernel_tiny_2d, upd2d)
    timings['tinygp_2d'] = time.perf_counter() - t0

    tf_fit = np.exp(np.asarray(kernel_tiny_2d.log_freqs))[:, 0]
    print('tinygp 2D  | loss:', round(float(loss2d), 3),
          ' | time:', round(timings['tinygp_2d'], 2), 's')
    print('  fitted time frequencies:', tf_fit, '-> periods:', 1.0 / tf_fit)
else:
    print('tinygp not installed.')


### `celerite2` — 2D limitation

`celerite2` is designed exclusively for 1D semiseparable covariance matrices; it does not
accept two-dimensional input coordinates. Attempting to pass `(time, wavelength)` data
raises an error:


In [ ]:
if HAVE_CELERITE:
    x2_c  = np.asarray(lc_2d.xdata.detach())   # shape (N, 2)
    y2_c  = np.asarray(lc_2d.ydata.detach())
    ye2_c = np.asarray(lc_2d.yerr.detach())

    kernel_c2d = c2terms.SHOTerm(sigma=0.7, rho=TRUE_PERIOD, tau=2*TRUE_PERIOD)
    gp_c2d = celerite2.GaussianProcess(kernel_c2d, mean=float(np.mean(y2_c)))
    try:
        gp_c2d.compute(x2_c, yerr=ye2_c)
        print('Unexpected: celerite2 accepted 2D input.')
    except Exception as exc:
        print('celerite2 rejects 2D input (expected):')
        print(f'  {type(exc).__name__}: {str(exc)[:200]}')
else:
    print('celerite2 not installed.')


## Summary comparison


In [ ]:
pgmuvi_2d_code = '''
res = lc_2d.fit(model='2D', num_mixtures=2, training_iter=150, lr=0.05)
'''

tinygp_2d_code = '''
# (1) define SMK2D class  (~15 lines, same as 1D but loop over 2 dims)
class SMK2D(TinyKernel):
    ...
    def evaluate(self, X1, X2):
        tau = X1 - X2
        return _dim(0) * _dim(1)   # product over dimensions
# (2) instantiate + run optax loop
'''

print('Code lines (2D fitting):')
print(f'  pgmuvi    :  {count_lines(pgmuvi_2d_code):2d}  (built-in 2D SMK shortcut)')
print(f'  tinygp    :  {count_lines(tinygp_2d_code):2d}+ (must hand-write 2D kernel class)')
print(f'  celerite2 :  N/A  (no native support for 2D / multiwavelength data)')
print()
print('Wall-clock times (seconds):')
for k, v in timings.items():
    print(f'  {k:<20s}: {v:.3f}')


## Practical guidance

| | `pgmuvi` | `tinygp` | `celerite2` |
|---|---|---|---|
| **1D SMK** | built-in | must define class | SHO proxy (Lorentzian) |
| **2D multiwavelength SMK** | built-in | must define class | **not supported** |
| **Optimiser** | Adam (PyTorch) | optax (JAX, user-provided) | scipy L-BFGS-B |
| **Spectral shape** | Gaussian | Gaussian | Lorentzian (SHOTerm) |
| **User code (1D)** | ~1 line | ~15 lines | ~5 lines |
| **User code (2D)** | ~1 line | ~20 lines | not applicable |

### When to choose each tool

- **`pgmuvi`** is the natural choice for astronomical multiwavelength light curves. Both 1D
  and 2D SMK models, MLS-based initialisation, and PyTorch-based optimisation are
  pre-configured behind a single `.fit()` call.
- **`celerite2`** should be preferred when ultra-fast $O(N)$ inference on a **single**
  time series is the priority and the exact spectral shape (Lorentzian vs Gaussian) is
  not critical. It is not suitable for multiwavelength analysis.
- **`tinygp`** is the right choice when you need **full flexibility** in kernel design
  and are comfortable writing JAX-based kernel classes and optimisation loops. It can
  reproduce any kernel that `pgmuvi` uses, but requires substantially more user code.
